In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

emails_full = pd.read_csv("./dataset.csv")

X = emails_full.drop(columns=["isSpam"])
y = emails_full["isSpam"]

print("duplicates before clean",emails_full["body"].duplicated().sum())

# Clean from duplicates
emails_full = emails_full.drop_duplicates(
    subset=["body"],
    keep="first"
).reset_index(drop=True)

# Recreate X and Y

X = emails_full.drop(columns=["isSpam"])
y = emails_full["isSpam"]

print("duplicates after clean", emails_full["body"].duplicated().sum())

# Splitting Train and Test Data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

duplicates before clean 3227
duplicates after clean 0


In [5]:
# Vectorizer
from sklearn.feature_extraction.text import CountVectorizer
import string

def cleanTexts(texts: list[str]):
		return [
			text.lower().translate(str.maketrans("", "", string.punctuation))
			for text in texts
		]

vectorizer = CountVectorizer()

X_train_vectors = vectorizer.fit_transform(
    cleanTexts(X_train["body"])
)

X_test_vectors = vectorizer.transform(
    cleanTexts(X_test["body"])
)

In [6]:
# Try logisticRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vectors, y_train)
logistic_scores = cross_val_score(model, X_train_vectors, y_train,
    scoring="accuracy", cv=10)

pd.Series(logistic_scores).describe()


count    10.000000
mean      0.979596
std       0.005182
min       0.971429
25%       0.976020
50%       0.980612
75%       0.981661
max       0.987755
dtype: float64

In [ ]:
# Evaluating the Model on Test Data
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

y_pred = model.predict(X_test_vectors)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
#                Predicted False         Predicted True
# Actual False   859                     6
# Actual True    17                      344
# Всего ошибок 17 + 6 = 23
print(confusion_matrix(y_test, y_pred))

0.9812398042414355
              precision    recall  f1-score   support

       False       0.98      0.99      0.99       865
        True       0.98      0.95      0.97       361

    accuracy                           0.98      1226
   macro avg       0.98      0.97      0.98      1226
weighted avg       0.98      0.98      0.98      1226

[[859   6]
 [ 17 344]]
